In [1]:
import pandas as pd
import os
import ast
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
import datetime
import requests

In [2]:
# Read in demand profiles that have been processed by the EIA data cleaner
# imputation script
regional_profiles = {}
hourly_demand_path = "../eia_data_cleaner/data/regional/outputs"
for fname in os.listdir(hourly_demand_path):
    fpath = os.path.join(hourly_demand_path, fname)
    if os.path.isdir(fpath):
        continue
    df = (
        pd.read_csv(fpath, parse_dates=["date_time"])
        .rename(columns={'date_time': 'timestamp', 'cleaned demand (MW)': 'value'})
        .set_index("timestamp")
        [["value"]]
    )
    region = fname.replace('.csv', '')
    regional_profiles[region] = df

subregional_profiles = {}
hourly_subregion_demand_path = "../eia_data_cleaner/data/subregional/outputs"
for fname in os.listdir(hourly_subregion_demand_path):
    fpath = os.path.join(hourly_subregion_demand_path, fname)
    if os.path.isdir(fpath):
        continue
    df = (
        pd.read_csv(fpath, parse_dates=["date_time"])
        .rename(columns={'date_time': 'timestamp', 'cleaned demand (MW)': 'value'})
        .set_index("timestamp")
        [["value"]]
    )
    subregion = fname.replace('.csv', '')
    subregional_profiles[subregion] = df

regions = list(regional_profiles.keys())
subregions = list(subregional_profiles.keys())

In [3]:
df_caiso = pd.read_csv("../data/iso_load_profiles/caiso.csv", parse_dates=['timestamp'])
df_pjm = pd.read_csv("../data/iso_load_profiles/pjm.csv", parse_dates=['timestamp'])

In [4]:
df_pgae_replacement = (
    df_caiso.loc[df_caiso.subba == 'PGAE']
    .set_index('timestamp')
    .sort_index()
    [['value']]
)
df_pgae_replacement = df_pgae_replacement.loc[(
    (df_pgae_replacement.index >= '2019-01-01 00:00:00')
    & (df_pgae_replacement.index <= '2019-05-02 07:00:00')
)]

pgae = subregional_profiles['PGAE'].copy()
pgae = (
    pd.concat([
        pgae.drop(df_pgae_replacement.index),
        df_pgae_replacement
    ])
    .sort_index()
)

subregional_profiles['PGAE'] = pgae

In [5]:
df_vea_replacement = (
    df_caiso.loc[df_caiso.subba == 'VEA']
    .set_index('timestamp')
    .sort_index()
    [['value']]
)
df_vea_replacement = df_vea_replacement.loc[(
    (df_vea_replacement.index >= '2019-12-23 09:00:00')
    & (df_vea_replacement.index <= '2020-05-02 07:00:00')
)]

vea = subregional_profiles['VEA'].copy()
vea = (
    pd.concat([
        vea.drop(df_vea_replacement.index),
        df_vea_replacement
    ])
    .sort_index()
)

subregional_profiles['VEA'] = vea

In [6]:
df_pl_replacement = (
    df_pjm.loc[df_pjm.subba == 'PL']
    .set_index('timestamp')
    .sort_index()
    [['value']]
)
df_pl_replacement = df_pl_replacement.loc[(
    (df_pl_replacement.index.year == 2019)
    & (df_pl_replacement.index.month == 1)
)]

pl = subregional_profiles['PL'].copy()
pl = (
    pd.concat([
        pl.drop(df_pl_replacement.index),
        df_pl_replacement
    ])
    .sort_index()
)

subregional_profiles['PL'] = pl

In [7]:
# Read in processed demand forecast profiles, which are used to correct
# the load shed portions of the relevant demand profiles
forecast_profiles = {}
hourly_forecast_path = "../eia_data_cleaner/data/forecast/outputs"
for fname in os.listdir(hourly_forecast_path):
    fpath = os.path.join(hourly_forecast_path, fname)
    if os.path.isdir(fpath):
        continue
    df = (
        pd.read_csv(fpath, parse_dates=["date_time"])
        .rename(columns={'date_time': 'timestamp', 'cleaned demand (MW)': 'value'})
        .set_index("timestamp")
        [["value"]]
    )
    region = fname.replace('.csv', '')
    forecast_profiles[region] = df

In [8]:
# Create name mappings for EIA-930 respondents, subregions, etc.
hourly_rto_demand = pd.read_csv(
    "../data/baseline_load_profiles/raw/2016_hourly_demand_by_rto.csv"
)
respondent_name_map = dict(zip(
    hourly_rto_demand['respondent'],
    hourly_rto_demand['respondent-name']
))

hourly_subregion_demand = pd.read_csv(
    "../data/baseline_load_profiles/raw/2016_hourly_demand_by_subregion.csv",
    dtype={'subba': str}
)
subba_name_map = dict(zip(
    hourly_subregion_demand['subba'], hourly_subregion_demand['subba-name']
))
parent_name_map = dict(zip(
    hourly_subregion_demand['parent'], hourly_subregion_demand['parent-name']
))

In [9]:
load_loss_events = pd.read_csv(
    "../data/load_loss_events.csv",
    parse_dates=['start_time', 'end_time']
)
load_loss_events = load_loss_events.sort_values(['start_time', 'end_time'])
load_loss_events['eia_codes'] = (
    load_loss_events['eia_codes']
    .apply(ast.literal_eval)
)
load_loss_events['ba_codes'] = (
    load_loss_events['ba_codes']
    .apply(ast.literal_eval)
)

In [10]:
events_by_eia_code = {}
eia_codes = set(sum(list(load_loss_events['eia_codes']), []))
for eia_code in eia_codes:
    all_events = []
    load_loss_events_sub = (
        load_loss_events.loc[(load_loss_events['eia_codes'].apply(lambda x: eia_code in x))]
    )
    for _, event in load_loss_events_sub.iterrows():
        event_start = event['start_time']
        event_end = event['end_time']
        if len(all_events) == 0:
            all_events.append((event_start, event_end))
        else:
            last_event_start, last_event_end = all_events[-1]
            if event_start > last_event_end:
                all_events.append((event_start, event_end))
            elif event_end > last_event_end:
                all_events[-1] = (last_event_start, event_end)
            else:
                pass
    events_by_eia_code[eia_code] = all_events

events_by_ba_code = {}
ba_codes = set(sum(list(load_loss_events['ba_codes']), []))
for ba_code in ba_codes:
    all_events = []
    load_loss_events_sub = (
        load_loss_events.loc[(load_loss_events['ba_codes'].apply(lambda x: ba_code in x))]
    )
    for _, event in load_loss_events_sub.iterrows():
        event_start = event['start_time']
        event_end = event['end_time']
        if len(all_events) == 0:
            all_events.append((event_start, event_end))
        else:
            last_event_start, last_event_end = all_events[-1]
            if event_start > last_event_end:
                all_events.append((event_start, event_end))
            elif event_end > last_event_end:
                all_events[-1] = (last_event_start, event_end)
            else:
                pass
    events_by_ba_code[ba_code] = all_events

In [11]:
# Helper functions for getting and plotting data
def get_load_profile(eia_code):
    if eia_code in regions:
        load_profile = regional_profiles[eia_code].copy()
    elif eia_code in subregions:
        load_profile = subregional_profiles[eia_code].copy()
    else:
        raise FileNotFoundError(f"ERROR: {eia_code} load not found.")
        
    return load_profile

def get_forecast_profile(eia_code):
    if eia_code in forecast_profiles.keys():
        forecast_profile = forecast_profiles[eia_code]
        return forecast_profile
    else:
        print(f"WARNING: {eia_code} forecast not found.")
        return None    

def get_event_load_profile(load_profile, event, margin_hours=0):
    event_start = event[0] - pd.Timedelta(hours=margin_hours)
    event_end = event[1] + pd.Timedelta(hours=margin_hours)
    event_load_profile = (
        load_profile.loc[(
            (load_profile.index >= event_start)
            & (load_profile.index <= event_end)
        )]
    )
    
    return event_load_profile

def remove_load_loss_events(load_profile, events, replace_with_na=False):
    for event in events:
        event_start, event_end = event
        if replace_with_na:
            load_profile.loc[(
                (load_profile.index >= event_start)
                & (load_profile.index <= event_end)
            ), 'value'] = np.nan
        else: 
            load_profile = load_profile.loc[(
                (load_profile.index < event_start)
                | (load_profile.index > event_end)
            )].copy()

    return load_profile

def plot_event_load_profile(load_profile, event, margin_hours=24, include_event_lines=True):
    f, ax = plt.subplots(figsize=(6, 6))
    event_load_profile = get_event_load_profile(load_profile, event, margin_hours)
    event_load_profile.plot(
        ax=ax,
        title=eia_code,
        linestyle='--',
        color='blue',
        xlabel="Datetime",
        ylabel="Load (MWh)",
    )
    ax.legend(['Original', 'Corrected'])
    if include_event_lines:
        ax.axvline(
            event[0],
            color='grey',
            linestyle='--',
            alpha=0.5
        )
        ax.axvline(
            event[1],
            color='grey',
            linestyle='--',
            alpha=0.5
        )

In [12]:
def predict_load_profile(eia_code, return_updated_load_profile=True):
    forecast_profile = get_forecast_profile(eia_code)
    if forecast_profile is None:
        load_profile = get_load_profile(eia_code)
        events = events_by_eia_code[eia_code]
        df = remove_load_loss_events(load_profile, events, replace_with_na=True)
        
        return df
    
    events = events_by_eia_code.get(eia_code, events_by_ba_code.get(eia_code))
    forecast_profile = forecast_profile.dropna()
    forecast_profile_no_events = remove_load_loss_events(forecast_profile, events)
    
    load_profile = get_load_profile(eia_code).dropna()
    load_profile_no_events = remove_load_loss_events(load_profile, events)
    
    intersection_index = forecast_profile_no_events.index.intersection(load_profile_no_events.index)
    forecast_profile_no_events = forecast_profile_no_events.loc[intersection_index]
    load_profile_no_events = load_profile_no_events.loc[intersection_index]
    
    X = np.array(forecast_profile_no_events.value).reshape(-1, 1)
    y = np.array(load_profile_no_events.value)
    reg = LinearRegression().fit(X, y)
    predicted_values = reg.predict(
        np.array(forecast_profile.value)
        .reshape(-1, 1)
    )
    predicted_load_profile = (
        pd.Series(predicted_values)
        .round(0)
        .astype(int)
        .reset_index()
        .set_index(forecast_profile.index)
        .drop(columns='index')
        .rename(columns={0: 'value'})
    )

    if return_updated_load_profile:
        load_profile_upd = load_profile.copy()
        for event in events:
            predicted_event_load_profile = predicted_load_profile.loc[(
                (predicted_load_profile.index >= event[0])
                & (predicted_load_profile.index <= event[1])
            )]
            load_profile_upd = (
                pd.concat([load_profile_upd, predicted_event_load_profile])
                .reset_index()
                .sort_values(['timestamp', 'value'], ascending=[True, False])
                .drop_duplicates(subset='timestamp', keep='first')
                .set_index('timestamp')
            )

            if eia_code in [
                'EAST',
                'COAS',
                'NRTH',
                'SOUT',
                'SCEN',
                'NCEN',
                'WEST',
                'FWES'
            ]:
                load_profile_upd.loc[(
                    (load_profile_upd.index >= event[0])
                    & (load_profile_upd.index <= event[1])
                    & (load_profile_upd.index < '6/30/2017 06:00:00')
                ), 'value'] = np.nan

            if eia_code in [
                '0001',
                '0027',
                '0035',
                '0004',
                '0006',
                '8910'
            ]:
                load_profile_upd.loc[(
                    (load_profile_upd.index >= event[0])
                    & (load_profile_upd.index <= event[1])
                    & (load_profile_upd.index < '8/30/2022 06:00:00')
                ), 'value'] = np.nan
                
        return load_profile_upd
    else:
        return predicted_load_profile

In [13]:
data_cleaner_inputs_fpath = '../eia_data_cleaner/data/load_loss_correction/raw_inputs'
os.makedirs(data_cleaner_inputs_fpath, exist_ok=True)

stat_collector = {}

respondents = sum([regions, subregions], [])
for eia_code in respondents:
    if (eia_code in events_by_eia_code.keys()) or (eia_code in events_by_ba_code.keys()):
        df = predict_load_profile(eia_code)
    else:
        df = get_load_profile(eia_code)

    if eia_code == 'NWMT':
        df.loc[(
            (df.index.year == 2021)
            & (df.index.month == 5)
            & (df.index.day.isin([14, 15, 16, 17, 18, 19, 20, 21, 22, 23]))
        ), 'value'] = np.nan

    stat_collector[eia_code] = df
    
    df = (
        df.reset_index()
        .rename(columns={'timestamp': 'date_time', 'value': 'demand (MW)'})
    )
    df.to_csv(
        os.path.join(data_cleaner_inputs_fpath, f"{eia_code}.csv"),
        index=False
    )
    print(f"Exported {eia_code}.csv")

Exported AEC.csv
Exported AECI.csv
Exported AVA.csv
Exported AZPS.csv
Exported BANC.csv
Exported BPAT.csv
Exported CHPD.csv
Exported CISO.csv
Exported CPLE.csv
Exported CPLW.csv
Exported DOPD.csv
Exported DUK.csv
Exported EPE.csv
Exported ERCO.csv
Exported FMPP.csv
Exported FPC.csv
Exported FPL.csv
Exported GCPD.csv
Exported GVL.csv
Exported HST.csv
Exported IID.csv
Exported IPCO.csv
Exported ISNE.csv
Exported JEA.csv
Exported LDWP.csv
Exported LGEE.csv
Exported MISO.csv
Exported NEVP.csv
Exported NSB.csv
Exported NWMT.csv
Exported NYIS.csv
Exported PACE.csv
Exported PACW.csv
Exported PGE.csv
Exported PJM.csv
Exported PNM.csv
Exported PSCO.csv
Exported PSEI.csv
Exported SC.csv
Exported SCEG.csv
Exported SCL.csv
Exported SOCO.csv
Exported SPA.csv
Exported SRP.csv
Exported SWPP.csv
Exported TAL.csv
Exported TEC.csv
Exported TEPC.csv
Exported TIDC.csv
Exported TPWR.csv
Exported TVA.csv
Exported WACM.csv
Exported WALC.csv
Exported WAUW.csv
Exported 0001.csv
Exported 0004.csv
Exported 0006.

In [14]:
# Before running the blocks below, run "eia_data_cleaner" with profile_type set to "load_loss_correction"

In [15]:
loss_corrected_load_profiles = {}
hourly_demand_path = "../eia_data_cleaner/data/load_loss_correction/outputs"
for fname in os.listdir(hourly_demand_path):
    fpath = os.path.join(hourly_demand_path, fname)
    if os.path.isdir(fpath):
        continue
    df = (
        pd.read_csv(fpath, parse_dates=["date_time"])
        .rename(columns={'date_time': 'timestamp', 'cleaned demand (MW)': 'value'})
        .set_index("timestamp")
        [["value"]]
    )
    eia_code = fname.replace('.csv', '')
    loss_corrected_load_profiles[eia_code] = df

In [16]:
processed_load_profiles = {}
for eia_code in loss_corrected_load_profiles.keys():
    load_profile_upd = loss_corrected_load_profiles[eia_code]
    load_profile = get_load_profile(eia_code)

    events = events_by_eia_code.get(eia_code, events_by_ba_code.get(eia_code))
    if events is not None:
        for event in events:
            original_event_load_profile = load_profile.loc[(
                (load_profile.index >= event[0])
                & (load_profile.index <= event[1])
            )]
            load_profile_upd = (
                pd.concat([load_profile_upd, original_event_load_profile])
                .reset_index()
                .sort_values(['timestamp', 'value'], ascending=[True, False])
                .drop_duplicates(subset='timestamp', keep='first')
                .set_index('timestamp')
            )

    df = load_profile_upd.copy()
        
    if eia_code == '8910':
        _df = (
            df.loc[(df.index.year == 2021) & (df.index.month == 2) & (df.index.day.isin([15, 17]))]
        )
        _df = _df.groupby(_df.index.hour).mean().round().astype(int)
        values = _df['value'].tolist()
        
        df.loc[(
            (df.index.year == 2021)
            & (df.index.month == 2)
            & (df.index.day == 16)
        ), 'value'] = values

    processed_load_profiles[eia_code] = df

In [17]:
for eia_code, df in processed_load_profiles.items():
    df.to_csv(f"../data/baseline_load_profiles/processed/{eia_code}.csv")